# DP: Gradient Privacy with Opacus

**Library:** [Opacus](https://opacus.ai/) 1.6+ (Meta, Apache-2.0)  
**Install:** `pip install opacus`  
**Stage:** During training (gradient clipping + noise injection)

---

## What DP Does

Differential Privacy adds calibrated noise to model updates so that no single training record can significantly influence the output. This prevents membership inference, gradient leakage, and canary extraction attacks.

```
Per-sample gradients  -->  Clip to norm C  -->  Add noise N(0, sigma^2 * C^2)  -->  Aggregate
```

In [ ]:
import os, sys
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)
print(f'Working directory: {os.getcwd()}')

## Step 1: DP Presets

The platform provides three presets. If no preset is specified, `DP_STRONG` is applied (fail-closed).

In [ ]:
from fl_pets.dp import get_preset, DP_PRESETS, compute_epsilon

print(f'{"Preset":<15} {"sigma":>8} {"C":>6} {"eps@100":>12} {"eps@500":>12}')
print('-' * 55)
for name in DP_PRESETS:
    cfg = get_preset(name)
    s = cfg['noise_multiplier']
    c = cfg['max_grad_norm']
    e100 = compute_epsilon(s, sample_rate=0.01, steps=100)
    e500 = compute_epsilon(s, sample_rate=0.01, steps=500)
    print(f'{name:<15} {s:>8.1f} {c:>6.1f} {e100:>12.4f} {e500:>12.4f}')
print('(sample_rate=0.01, delta=1e-5)')

## Step 2: Wrap a Model with DP

Opacus wraps any PyTorch model + optimizer to enforce per-sample gradient clipping and noise injection.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from fl_pets.dp import make_private

# Simple model
model = nn.Sequential(nn.Linear(30, 64), nn.ReLU(), nn.Linear(64, 1), nn.Sigmoid())
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Synthetic data
X = torch.randn(200, 30)
y = (torch.randn(200) > 0).float().unsqueeze(1)
loader = DataLoader(TensorDataset(X, y), batch_size=32)

# Make private with DP_MODERATE preset
private_model, private_optimizer, private_loader = make_private(
    model, optimizer, loader, preset="DP_MODERATE", epochs=5
)
print(f'Model wrapped with DP_MODERATE (sigma=0.8, C=1.0)')
print(f'DP-SGD will clip per-sample gradients and add Gaussian noise each step')

## Step 3: Privacy Budget Calculator

Epsilon grows with training rounds. Use the budget calculator to plan training duration.

In [ ]:
from fl_pets.dp import compute_epsilon

print('Privacy budget (epsilon) vs training rounds:')
print(f'{"Rounds":>8}  {"DP_STRONG":>12}  {"DP_MODERATE":>12}  {"DP_RELAXED":>12}')
print('-' * 50)
for rounds in [10, 50, 100, 200, 500]:
    eps_s = compute_epsilon(1.5, sample_rate=0.01, steps=rounds)
    eps_m = compute_epsilon(0.8, sample_rate=0.01, steps=rounds)
    eps_r = compute_epsilon(0.5, sample_rate=0.01, steps=rounds)
    print(f'{rounds:>8}  {eps_s:>12.4f}  {eps_m:>12.4f}  {eps_r:>12.4f}')

## When DP Is Not the Right Tool

| Model size | DP impact | Recommendation |
|-----------|-----------|----------------|
| Small (<1M params): MLP, BiLSTM | Minimal accuracy loss | Use DP |
| Medium (1-8M): CNN1D, ResNet-small | Moderate accuracy loss | Use DP with `DP_RELAXED` or SecAgg |
| Large (>8M): DenseNet-121, LLMs | Accuracy drops to random | Use SecAgg or Federated LoRA instead |

## References

- [Opacus](https://opacus.ai/) — Meta, Apache-2.0
- Abadi et al. (2016), *Deep Learning with Differential Privacy*, CCS
- Mironov (2017), *Renyi Differential Privacy*, CSF

## Next Steps

- [SecAgg: Update Masking](secagg-update-masking.ipynb) — hide individual updates from the server
- [Tutorial 4](../intermediate/04-differential-privacy.ipynb) — DP in the full FL training loop